### Database Connectivity with SQLite

Objectives:
Understand the basics of relational databases and SQL.
Use Python's built-in sqlite3 module to connect to a database.
Perform CRUD (Create, Read, Update, Delete) operations.
Use parameterized queries to prevent SQL injection.

Class Outline:
Introduction to Databases (SQL vs NoSQL, SQLite).
Connecting to SQLite, Creating Tables.
CRUD Operations: Insert, Select, Update, Delete.
Mini-Project: Simple Inventory Management System.

1. Introduction to SQLite
SQLite is a serverless, self-contained, transactional SQL database engine. It's incredibly popular because it's simple to use 
and the entire database is stored in a single file on your disk. Python has a built-in module, sqlite3, so no installation is needed.
The typical workflow is:
Connect to the database (creates the file if it doesn't exist).
Create a Cursor object to execute commands.
Execute SQL queries.
Commit (save) the changes.
Close the connection.
2. Creating a Connection and a Table

In [1]:
import sqlite3

# 1. Connect to a database (it will be created if it doesn't exist)
conn = sqlite3.connect('university.db')

# 2. Create a cursor object
cursor = conn.cursor()

# 3. Execute a SQL command to create a table
# Use triple quotes for multi-line strings
cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        major TEXT,
        gpa REAL
    )
''')

# 4. Commit the changes
conn.commit()

print("Database and table created successfully.")

# 5. Close the connection
conn.close()


Database and table created successfully.


3. CRUD Operations
CRUD stands for Create, Read, Update, and Delete. These are the four basic functions of persistent storage.
C: Create (Insert Data)
IMPORTANT: Always use parameterized queries (?) to insert data. This prevents a security vulnerability called SQL Injection.

 

In [2]:
   
import sqlite3

conn = sqlite3.connect('university.db')
cursor = conn.cursor()

# Insert a single record
cursor.execute("INSERT INTO students (name, major, gpa) VALUES (?, ?, ?)", 
               ('Alice', 'Computer Science', 3.9))

# Insert multiple records
students_to_add = [
    ('Bob', 'Physics', 3.5),
    ('Charlie', 'Mathematics', 3.7)
]
cursor.executemany("INSERT INTO students (name, major, gpa) VALUES (?, ?, ?)", students_to_add)

conn.commit()
print(f"{cursor.rowcount} records inserted successfully.")
conn.close()

2 records inserted successfully.


In [ ]:
# R: Read (Select Data)

import sqlite3

conn = sqlite3.connect('university.db')
cursor = conn.cursor()

# Select all students
cursor.execute("SELECT * FROM students")
all_students = cursor.fetchall() # fetchall() gets all rows
print("All Students:")
for student in all_students:
    print(student)

# Select one specific student
print("\nFetching Bob's record:")
cursor.execute("SELECT * FROM students WHERE name = ?", ('Bob',))
bob = cursor.fetchone() # fetchone() gets the first matching row
print(bob)

conn.close()

In [3]:
# U: Update Data

import sqlite3

conn = sqlite3.connect('university.db')
cursor = conn.cursor()

# Update Charlie's GPA
cursor.execute("UPDATE students SET gpa = ? WHERE name = ?", (3.8, 'Charlie'))
conn.commit()

print(f"Rows updated: {cursor.rowcount}")

# Verify the update
cursor.execute("SELECT * FROM students WHERE name = 'Charlie'")
print(f"Charlie's new record: {cursor.fetchone()}")

conn.close()

Rows updated: 1
Charlie's new record: (3, 'Charlie', 'Mathematics', 3.8)


In [4]:
# D: Delete Data

import sqlite3

conn = sqlite3.connect('university.db')
cursor = conn.cursor()

# Let's add a temporary student to delete
cursor.execute("INSERT INTO students (name, major, gpa) VALUES (?, ?, ?)", ('David', 'Art', 2.5))
conn.commit()
print("Added David for deletion.")

# Delete the record
cursor.execute("DELETE FROM students WHERE name = ?", ('David',))
conn.commit()

print(f"Rows deleted: {cursor.rowcount}")
conn.close()

Added David for deletion.
Rows deleted: 1


Mini-Project: Simple Inventory Management System
Create a Python script with functions to manage a product inventory stored in an SQLite database (inventory.db).
The products table should have: id, name, category, price, quantity.
Implement functions for:
add_product(name, category, price, quantity)
view_all_products()
update_product_quantity(name, new_quantity)
delete_product(name)

In [5]:
import sqlite3

def setup_database():
    conn = sqlite3.connect('inventory.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL UNIQUE,
            category TEXT,
            price REAL,
            quantity INTEGER
        )
    ''')
    conn.commit()
    conn.close()

In [6]:
def add_product(name, category, price, quantity):
    conn = sqlite3.connect('inventory.db')
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO products (name, category, price, quantity) VALUES (?, ?, ?, ?)",
                       (name, category, price, quantity))
        conn.commit()
        print(f"Product '{name}' added successfully.")
    except sqlite3.IntegrityError:
        print(f"Error: Product '{name}' already exists.")
    finally:
        conn.close()

In [7]:
def view_all_products():
    conn = sqlite3.connect('inventory.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM products ORDER BY name")
    products = cursor.fetchall()
    conn.close()
    if not products:
        print("Inventory is empty.")
    else:
        print("\n--- Current Inventory ---")
        for p in products:
            print(f"ID: {p[0]}, Name: {p[1]}, Category: {p[2]}, Price: ${p[3]:.2f}, Quantity: {p[4]}")
        print("-------------------------\n")

In [8]:
def update_product_quantity(name, new_quantity):
    conn = sqlite3.connect('inventory.db')
    cursor = conn.cursor()
    cursor.execute("UPDATE products SET quantity = ? WHERE name = ?", (new_quantity, name))
    conn.commit()
    if cursor.rowcount == 0:
        print(f"Error: Product '{name}' not found.")
    else:
        print(f"Quantity for '{name}' updated to {new_quantity}.")
    conn.close()

In [9]:
def delete_product(name):
    conn = sqlite3.connect('inventory.db')
    cursor = conn.cursor()
    cursor.execute("DELETE FROM products WHERE name = ?", (name,))
    conn.commit()
    if cursor.rowcount == 0:
        print(f"Error: Product '{name}' not found.")
    else:
        print(f"Product '{name}' deleted.")
    conn.close()

In [10]:
# --- Main execution ---
setup_database()

# Add some initial products
add_product('Laptop', 'Electronics', 1200.00, 10)
add_product('Mouse', 'Electronics', 25.50, 50)
add_product('Book', 'Stationery', 15.00, 100)
add_product('Laptop', 'Electronics', 1200.00, 10) # Test duplicate error

# View current inventory
view_all_products()

# Update a product
update_product_quantity('Laptop', 8)
update_product_quantity('Keyboard', 20) # Test not found error

# Delete a product
delete_product('Book')

# View final inventory
view_all_products()

Product 'Laptop' added successfully.
Product 'Mouse' added successfully.
Product 'Book' added successfully.
Error: Product 'Laptop' already exists.

--- Current Inventory ---
ID: 3, Name: Book, Category: Stationery, Price: $15.00, Quantity: 100
ID: 1, Name: Laptop, Category: Electronics, Price: $1200.00, Quantity: 10
ID: 2, Name: Mouse, Category: Electronics, Price: $25.50, Quantity: 50
-------------------------

Quantity for 'Laptop' updated to 8.
Error: Product 'Keyboard' not found.
Product 'Book' deleted.

--- Current Inventory ---
ID: 1, Name: Laptop, Category: Electronics, Price: $1200.00, Quantity: 8
ID: 2, Name: Mouse, Category: Electronics, Price: $25.50, Quantity: 50
-------------------------

